# 0단계: 전사 표기 방식에 의한 데이터 누수 분석 및 정제

**핵심 문제**: phishing(FSS 전사)과 normal(AI Hub 전사)이 서로 다른 전사 규칙을 사용 →  
이 패턴 차이가 `label`과 거의 1:1 대응 → 모델이 '피싱 여부'가 아니라 '어느 기관이 전사했는가'를 학습

| 패턴 | 예시 | 발생 측 |
|---|---|---|
| 발화자·간투사 태그 | `n/`, `아/`, `b/`, `씁/` | normal(AI Hub)만 |
| 숫자 발음 주석 | `(80)/(팔순)` | normal(AI Hub)만 |
| 단어 정정 표시 | `+` | normal(AI Hub)만 |
| 말줄임 | `...` | phishing(FSS)만 |

## 0. 라이브러리 로드

In [ ]:
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

DATA_PATH = 'Data/metadata.jsonl'

## 1. 데이터 로딩

In [ ]:
records = []
with open(DATA_PATH, encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
print(f'총 행 수: {len(df)}')
print(f'label 분포:\n{df["label"].value_counts()}')
print(f'split 분포:\n{df["split"].value_counts()}')
df.head(2)

---
## 2. 컬럼별 용도 분류

각 컬럼의 역할을 분류하고, **학습 피처로 절대 사용 금지**인 컬럼을 데이터로 직접 검증합니다.

### 2-1. 컬럼 카탈로그

In [ ]:
catalog = [
    ('sample_id',          '고유 식별자',                      '미사용',     '결과 추적용 보관'),
    ('split',              'train/val/test 지정',              '분할 기준',  '재분할 금지 — 그대로 사용'),
    ('label',              'phishing/normal 텍스트 라벨',      '미사용',     'EDA·리포트 표시용'),
    ('label_id',           '1/0 이진 라벨',                    '타겟(y)',    '모델 학습 목적 변수'),
    ('parent_call_id',     '원본 통화 단위 ID',                '미사용',     '분할 시 그룹 기준으로 이미 반영, 검증용만'),
    ('source_id',          '세부 소스 ID',                     '미사용',     '추적용'),
    ('source_dataset',     'fss / aihub 출처',                 '사용 금지',  'label과 완전 대응 → 트리비얼 리키지'),
    ('source_audio_path',  'Windows 절대경로',                 '미사용',     '개인 PC 경로 — 마스킹 대상'),
    ('audio_path',         '상대 wav 경로',                    '미사용',     '텍스트 단계에서는 미사용, 오디오 임베딩 단계에서 사용'),
    ('start_sec',          '원본 통화 내 시작 위치(초)',        '미사용',     '모델 피처로 의미 없음'),
    ('end_sec',            '원본 통화 내 종료 위치(초)',        '미사용',     '모델 피처로 의미 없음'),
    ('duration_sec',       '세그먼트 길이(초)',                 '조건부',     'label 분포 차이 없을 때만 구조적 피처로 사용'),
    ('text',               '원본 전사문',                       '원천',       'text_clean 생성 후 직접 피처로 쓰지 않음'),
    ('qa_status',          'PASS / SOURCE_TEXT',               '사용 금지',  'label과 완전 대응 → 값 자체가 사실상 라벨'),
    ('qa_flags',           '전부 공백',                        '미사용',     '정보 없음'),
    ('is_augmented',       '전부 False',                       '미사용',     '정보 없음'),
    ('augmentation',       '전부 공백',                        '미사용',     '정보 없음'),
]

cat_df = pd.DataFrame(catalog, columns=['컬럼', '역할', '사용 여부', '비고'])

def highlight_usage(row):
    if row['사용 여부'] == '사용 금지':
        return ['background-color: #ffcccc'] * len(row)
    elif row['사용 여부'] in ('타겟(y)', '분할 기준', '원천'):
        return ['background-color: #cce5ff'] * len(row)
    elif row['사용 여부'] == '조건부':
        return ['background-color: #fff3cd'] * len(row)
    return [''] * len(row)

cat_df.style.apply(highlight_usage, axis=1)

### 2-2. 사용 금지 컬럼 검증 — `source_dataset` ↔ `label` 완전 대응

In [ ]:
cross_source = pd.crosstab(df['source_dataset'], df['label'])
print('=== source_dataset × label 교차표 ===')
print(cross_source)

# fss→phishing, aihub→normal 외의 조합이 0건이면 완전 대응
leak_source = cross_source.loc['fss', 'normal'] + cross_source.loc['aihub', 'phishing']
print(f'\n비정상 조합(fss+normal, aihub+phishing) 건수: {leak_source}')
print('→ source_dataset은 label과 완전 대응 — 학습 피처 절대 불가' if leak_source == 0
      else '→ 일부 크로스 존재 (확인 필요)')

### 2-3. 사용 금지 컬럼 검증 — `qa_status` ↔ `label` 완전 대응

In [ ]:
cross_qa = pd.crosstab(df['qa_status'], df['label'])
print('=== qa_status × label 교차표 ===')
print(cross_qa)

# PASS→phishing, SOURCE_TEXT→normal 외 조합이 0이면 완전 대응
vals = cross_qa.values
off_diag = vals.sum() - vals.diagonal().sum()
print(f'\n비대각 합(크로스 조합): {off_diag}')
print('→ qa_status는 label과 완전 대응 — 학습 피처 절대 불가' if off_diag == 0
      else '→ 일부 크로스 존재 (확인 필요)')

### 2-4. 상수 컬럼 검증 — `is_augmented`, `qa_flags`, `augmentation`

In [ ]:
const_checks = {
    'is_augmented (전부 False)': (df['is_augmented'].nunique() == 1) and (df['is_augmented'].iloc[0] == False),
    'qa_flags (전부 공백)':       (df['qa_flags'].fillna('').str.strip() == '').all(),
    'augmentation (전부 공백)':   (df['augmentation'].fillna('').str.strip() == '').all(),
}

print('=== 상수 컬럼 검증 ===')
for item, ok in const_checks.items():
    print(f'  [{"PASS" if ok else "FAIL"}] {item}')

### 2-5. 조건부 피처 검증 — `duration_sec` 분포 (label 간 차이 여부)

In [ ]:
phishing_dur = df.loc[df['label'] == 'phishing', 'duration_sec']
normal_dur   = df.loc[df['label'] == 'normal',   'duration_sec']

t_stat, p_val = stats.ttest_ind(phishing_dur, normal_dur)

print('=== duration_sec 기술통계 (label별) ===')
print(df.groupby('label')['duration_sec'].describe().round(2))
print(f'\nt-검정: t={t_stat:.3f}, p={p_val:.4f}')
if p_val < 0.05:
    print('→ label 간 분포 차이가 통계적으로 유의 (p<0.05) — 리키지 후보, 피처 제외 또는 별도 검토 필요')
else:
    print('→ label 간 분포 차이 없음 (p≥0.05) — 구조적 피처로 사용 가능')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 히스토그램
axes[0].hist(phishing_dur, bins=30, alpha=0.6, label='phishing', color='#DD8452')
axes[0].hist(normal_dur,   bins=30, alpha=0.6, label='normal',   color='#4C72B0')
axes[0].set_title('duration_sec 분포 (label별)')
axes[0].set_xlabel('초(sec)')
axes[0].legend()
axes[0].text(0.97, 0.95, f'p={p_val:.4f}', transform=axes[0].transAxes,
             ha='right', va='top', fontsize=10,
             color='red' if p_val < 0.05 else 'green')

# 박스플롯
axes[1].boxplot([phishing_dur, normal_dur], labels=['phishing', 'normal'],
                patch_artist=True,
                boxprops=dict(facecolor='#f0f0f0'))
axes[1].set_title('duration_sec 박스플롯')
axes[1].set_ylabel('초(sec)')

plt.tight_layout()
plt.show()

### 2-6. 사용 컬럼 최종 정리

In [ ]:
print('=' * 55)
print('컬럼 용도 최종 정리')
print('=' * 55)
print('  [타겟(y)]   label_id')
print('  [피처(X)]   text_clean  (text 정제 후 생성)')
print('  [분할 기준] split')
print('  [추적용]    sample_id, parent_call_id')
print()
print('  [사용 금지] source_dataset  ← label과 완전 대응')
print('  [사용 금지] qa_status       ← label과 완전 대응')
print()
dur_note = '조건부 가능 (p≥0.05)' if p_val >= 0.05 else '리키지 후보 — 별도 검토 필요'
print(f'  [조건부]    duration_sec    ← {dur_note}')
print('=' * 55)

---
## 3. 누수 패턴 존재 확인 (정제 전)

In [ ]:
def detect_patterns(text):
    speaker_tag  = bool(re.search(r'(?:^|\s)[a-zA-Z가-힣]{1,2}/', text))
    number_annot = bool(re.search(r'\([^)]*\)/\([^)]*\)', text))
    correction   = bool(re.search(r'\+', text))
    ellipsis     = bool(re.search(r'\.{3}', text))
    return speaker_tag, number_annot, correction, ellipsis

leak_cols = ['has_speaker_tag', 'has_number_annot', 'has_correction', 'has_ellipsis']
df[leak_cols] = df['text'].apply(lambda t: pd.Series(detect_patterns(t)))

summary = df.groupby('label')[leak_cols].sum().T
summary.columns.name = None
summary.index.name = '패턴'
print('=== 정제 전 패턴 발생 건수 ===')
print(summary)

In [ ]:
pattern_labels = ['발화자 태그\n(n/, 아/ …)', '숫자 주석\n((80)/(팔순))', '단어 정정\n(+)', '말줄임\n(...)']
colors = ['#4C72B0', '#DD8452']

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, col, label in zip(axes, leak_cols, pattern_labels):
    counts = df.groupby('label')[col].sum()
    ax.bar(counts.index, counts.values, color=colors)
    ax.set_title(label, fontsize=11)
    ax.set_ylabel('발생 건수')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 5, str(int(v)), ha='center', fontsize=10)

fig.suptitle('정제 전: label별 전사 규칙 패턴 발생 건수', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3-1. 실제 예시 확인

In [ ]:
def show_examples(col, label_val, n=3):
    mask = (df[col] == True) & (df['label'] == label_val)
    samples = df.loc[mask, 'text'].head(n)
    print(f'\n[{col} | label={label_val}] 예시 {len(samples)}건')
    for i, t in enumerate(samples, 1):
        print(f'  {i}. {t[:120]}')

show_examples('has_speaker_tag', 'normal')
show_examples('has_number_annot', 'normal')
show_examples('has_correction', 'normal')
show_examples('has_ellipsis', 'phishing')

## 4. 텍스트 정제 함수 구현 (text → text_clean)

In [ ]:
def clean_text(text: str) -> str:
    """AI Hub / FSS 전사 규칙 차이를 제거해 label 누수를 방지한다."""

    # 1) AI Hub 숫자 주석: (80)/(팔순) → 팔순
    text = re.sub(r'\([^)]*\)/\(([^)]*)\)', r'\1', text)

    # 2) 발화자·간투사 태그: 공백/줄 시작 뒤 '1~2글자/' 제거
    text = re.sub(r'(?:(?<=\s)|(?<=^))[a-zA-Z가-힣]{1,2}/\s*', '', text)

    # 3) 단어 정정 '+': '+' 앞 불완전 발화 제거
    text = re.sub(r'\S+\+\s*', '', text)

    # 4) 말줄임 '...': 공통 토큰 <PAUSE> 로 정규화
    text = re.sub(r'\.{3,}', ' <PAUSE> ', text)

    # 5) 공백 정리
    text = re.sub(r'[ \t]+', ' ', text).strip()

    return text


tests = [
    ('숫자 주석', '(80)/(팔순) 잔치에 오셨습니다'),
    ('발화 태그', 'n/ 아/ 거기 사시는 거예요 b/ 네 맞아요'),
    ('단어 정정', '할아버+ 할머니께서 오셨습니다'),
    ('말줄임',   '그래서요... 잠깐만요... 확인해볼게요'),
    ('복합',     'n/ 아/ (80)/(팔순) 할아버+ 할머니... 오셨어요'),
]

print('=== 정제 함수 단위 테스트 ===')
for name, t in tests:
    print(f'[{name}]')
    print(f'  원본  : {t}')
    print(f'  정제후: {clean_text(t)}')
    print()

## 5. 전체 데이터에 정제 적용

In [ ]:
df['text_clean'] = df['text'].apply(clean_text)

after_cols = ['has_speaker_tag_c', 'has_number_annot_c', 'has_correction_c', 'has_ellipsis_c']
df[after_cols] = df['text_clean'].apply(lambda t: pd.Series(detect_patterns(t)))

summary_after = df.groupby('label')[after_cols].sum().T
summary_after.columns.name = None
summary_after.index = leak_cols

print('=== 정제 후 패턴 잔존 건수 (0이어야 함) ===')
print(summary_after)

## 6. 정제 전/후 비교 시각화

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for col_idx, (before_col, after_col, plabel) in enumerate(zip(
    leak_cols, after_cols, pattern_labels
)):
    for row_idx, (col, title) in enumerate([
        (before_col, '정제 전'),
        (after_col,  '정제 후'),
    ]):
        ax = axes[row_idx][col_idx]
        counts = df.groupby('label')[col].sum()
        bars = ax.bar(counts.index, counts.values, color=colors)
        ax.set_title(f'{plabel}\n({title})', fontsize=9)
        ax.set_ylim(0, summary.values.max() * 1.15)
        for bar, v in zip(bars, counts.values):
            ax.text(bar.get_x() + bar.get_width()/2, v + 5, str(int(v)),
                    ha='center', fontsize=9)

fig.suptitle('전사 규칙 패턴 발생 건수: 정제 전(위) vs 정제 후(아래)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. 정제로 인한 텍스트 길이 변화 확인

In [ ]:
df['text_len']       = df['text'].str.len()
df['text_clean_len'] = df['text_clean'].str.len()
df['len_diff']       = df['text_len'] - df['text_clean_len']

print('=== 텍스트 길이 변화 (label별) ===')
print(df.groupby('label')[['text_len', 'text_clean_len', 'len_diff']].mean().round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, label_val in zip(axes, ['phishing', 'normal']):
    sub = df[df['label'] == label_val]
    ax.hist(sub['text_len'],       bins=40, alpha=0.6, label='원본')
    ax.hist(sub['text_clean_len'], bins=40, alpha=0.6, label='정제후')
    ax.set_title(f'{label_val} — 텍스트 길이 분포')
    ax.set_xlabel('글자 수')
    ax.legend()

plt.tight_layout()
plt.show()

## 8. 검증 체크리스트

In [ ]:
checks = {}

residual = summary_after.sum(axis=1)
for pat, cnt in residual.items():
    checks[f'패턴 잔존 0: {pat}'] = (cnt == 0, int(cnt))

checks['source_dataset ↔ label 완전 대응 확인'] = (leak_source == 0, f'크로스 {leak_source}건')
checks['qa_status ↔ label 완전 대응 확인']      = (off_diag == 0,    f'크로스 {off_diag}건')
checks['source_dataset 피처 미포함']             = (True, 'text_clean만 사용')
checks['qa_status 피처 미포함']                  = (True, 'text_clean만 사용')
checks['text_clean 결측 없음']                   = (df['text_clean'].isna().sum() == 0,
                                                    int(df['text_clean'].isna().sum()))

print('=== 검증 체크리스트 ===')
all_pass = True
for item, (ok, detail) in checks.items():
    status = 'PASS' if ok else 'FAIL'
    if not ok:
        all_pass = False
    print(f'  [{status}] {item}  ({detail})')

print()
print('모든 체크 통과!' if all_pass else '일부 체크 실패 — 정제 로직 보완 필요')

## 9. 정제된 데이터 저장

In [ ]:
save_cols = [
    'sample_id', 'split', 'label', 'label_id',
    'parent_call_id', 'audio_path', 'duration_sec',
    'text', 'text_clean'
]
df[save_cols].to_json(
    'Data/metadata_clean.jsonl',
    orient='records', lines=True, force_ascii=False
)
print(f'저장 완료: Data/metadata_clean.jsonl ({len(df)}행)')
print('이후 단계에서는 text_clean 컬럼을 핵심 피처(X)로 사용할 것')